In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results__.html
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/submission.csv
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__notebook__.ipynb
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__output__.json
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/model.png
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/custom.css
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results___files/__results___24_0.png
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results___files/__results___27_0.png
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results___files/__results___30_0.png
/kaggle/input/notebooks/chiragsharma1201/digital-recognizer/__results___files/__results___28_0.png
/kaggle/input/notebooks/chiragsharma1201/predict-introverts-from-extroverts/__results__.html
/kaggle/input/notebooks/chiragsharma1201/predict-introverts-fro

In [2]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate bitsandbytes
!pip install -q tree-sitter==0.20.4 tree-sitter-languages einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.9/493.9 kB 8.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 81.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import re
import os
import faiss
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
from tree_sitter import Parser
from tree_sitter_languages import get_language
from kaggle_secrets import UserSecretsClient

In [4]:
secrets = UserSecretsClient()
hf_token = secrets.get_secret("hugging_face")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(" Loading Mistral-7B locally (~3-4 mins)...")

tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    token=hf_token
)
llm_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

tokenizer.pad_token = tokenizer.eos_token
llm_model.config.pad_token_id = llm_model.config.eos_token_id

print(" Mistral-7B loaded!")

 Loading Mistral-7B locally (~3-4 mins)...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

 Mistral-7B loaded!


In [5]:
print(" Loading BAAI/bge-base-en-v1.5 embedder...")
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

print(" Loading cross-encoder reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print(" Embedder + Reranker ready!")

 Loading BAAI/bge-base-en-v1.5 embedder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

 Embedder + Reranker ready!


In [6]:
LANG_MAP = {
    ".py":   "python",
    ".js":   "javascript",
    ".ts":   "typescript",
    ".jsx":  "javascript",
    ".tsx":  "typescript",
    ".java": "java",
    ".cpp":  "cpp",
    ".cc":   "cpp",
    ".c":    "c",
    ".go":   "go",
    ".rs":   "rust",
    ".rb":   "ruby",
    ".cs":   "c_sharp",
}

CHUNK_NODE_TYPES = {
    "python":     ["function_definition", "class_definition"],
    "javascript": ["function_declaration", "class_declaration",
                   "arrow_function", "method_definition"],
    "typescript": ["function_declaration", "class_declaration",
                   "arrow_function", "method_definition",
                   "interface_declaration", "type_alias_declaration"],
    "java":       ["method_declaration", "class_declaration",
                   "interface_declaration"],
    "cpp":        ["function_definition", "class_specifier"],
    "c":          ["function_definition"],
    "go":         ["function_declaration", "method_declaration",
                   "type_declaration"],
    "rust":       ["function_item", "impl_item", "struct_item",
                   "enum_item", "trait_item"],
    "ruby":       ["method", "class", "module"],
    "c_sharp":    ["method_declaration", "class_declaration",
                   "interface_declaration"],
}
IMPORT_NODE_TYPES = {
    "import_statement",        
    "import_from_statement",   
    "import_declaration",      
    "using_directive",        
    "include_statement",      
}

In [7]:
parser = Parser()

def get_node_text(src_bytes, node):
    return src_bytes[node.start_byte:node.end_byte].decode("utf-8", errors="replace")


def extract_name(src_bytes, node, lang):
    """Best-effort function/class name extraction across languages."""
    for field in ("name", "declarator"):
        n = node.child_by_field_name(field)
        if n:
            return get_node_text(src_bytes, n).split("(")[0].strip()
    for child in node.children:
        if child.is_named and child.type not in ("comment", "block"):
            txt = get_node_text(src_bytes, child).split("(")[0].strip()
            if txt:
                return txt
    return "<anonymous>"

In [8]:
def extract_chunks_treesitter(src_code, lang_name):
    """Parse source with Tree-sitter and extract named code units."""
    target_types = set(CHUNK_NODE_TYPES.get(lang_name, []))
    src_bytes = src_code.encode("utf-8", errors="replace")
    tree = parser.parse(src_bytes)

    chunks = []
    imports = []

    def traverse(node):
        t = node.type

        if t in IMPORT_NODE_TYPES:
            imports.append({
                "name": get_node_text(src_bytes, node).strip()[:120],
                "type": "import",
                "code": get_node_text(src_bytes, node),
                "start": node.start_point,
                "end": node.end_point,
            })
            return  #
        if t in target_types:
            name = extract_name(src_bytes, node, lang_name)
            chunks.append({
                "name": name,
                "type": t,
                "code": get_node_text(src_bytes, node),
                "start": node.start_point,
                "end": node.end_point,
            })
          

        for child in node.children:
            traverse(child)

    traverse(tree.root_node)
    return chunks, imports
    

def chunk_by_sliding_window(text, chunk_size=40, overlap=10):
    """Split plain text into overlapping line-based windows."""
    lines = text.splitlines()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, max(1, len(lines) - overlap), step):
        window = lines[i: i + chunk_size]
        if not window:
            break
        chunks.append({
            "name": f"lines_{i+1}_{i+len(window)}",
            "type": "text_chunk",
            "code": "\n".join(window),
            "start": (i, 0),
            "end": (i + len(window), 0),
        })
    return chunks


In [9]:
def load_chunks_from_file(file_path):
    """Auto-detect language and extract chunks from any supported file."""
    # FIX: File existence check
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return [], "unknown"

    ext = os.path.splitext(file_path)[1].lower()
    lang_name = LANG_MAP.get(ext)

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        source = f.read()

    if lang_name:
        try:
            lang = get_language(lang_name)
            parser.set_language(lang)
            code_chunks, imports = extract_chunks_treesitter(source, lang_name)
            all_chunks = code_chunks + imports
            print(f" Language: {lang_name} | Code chunks: {len(code_chunks)} | Imports: {len(imports)}")
            return all_chunks, lang_name
        except Exception as e:
            print(f"Tree-sitter failed for {lang_name}: {e} — falling back to text chunking")

    chunks = chunk_by_sliding_window(source)
    print(f" Plain-text fallback | Chunks: {len(chunks)}")
    return chunks, "plaintext"


print(" Parser utilities ready!")

 Parser utilities ready!


In [10]:
print("Enter the path to your file (any language):")
file_path = input(" File path: ").strip()
all_chunks, detected_lang = load_chunks_from_file(file_path)
seen = set()
all_chunks = [c for c in all_chunks if not (c["name"] in seen or seen.add(c["name"]))]
print(f"\nTotal chunks loaded: {len(all_chunks)}")
for c in all_chunks[:5]:
    print(f"  [{c['type']}] {c['name']}")

Enter the path to your file (any language):


 File path:  /kaggle/input/datasets/chiragsharma1201/code-compressor/cc_from_graph_and_ss.py


 Language: python | Code chunks: 20 | Imports: 17

Total chunks loaded: 33
  [function_definition] get_code_text
  [function_definition] add_node
  [function_definition] add_edge
  [function_definition] extract_chunks_and_graph
  [function_definition] get_name


In [11]:
code_texts = [
    f"Type: {c['type']}\nName: {c['name']}\n\n{c['code']}"
    for c in all_chunks
]

embeddings = embedder.encode(
    code_texts,
    show_progress_bar=True,
    batch_size=16,
    normalize_embeddings=True  
)
print(f" Embeddings shape: {embeddings.shape}")


dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(np.array(embeddings, dtype=np.float32))

print(f"FAISS index ready with {index.ntotal} chunks")

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

 Embeddings shape: (33, 768)
FAISS index ready with 33 chunks


In [12]:

def call_mistral(system_prompt, user_prompt, max_tokens=800):
    """Call Mistral-7B with increased context window."""
    prompt = f"<s>[INST] {system_prompt}\n\n{user_prompt} [/INST]"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=8192  
    ).to("cuda")

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [13]:
def retrieve_and_rerank(query, top_k=10, final_k=5):
    """
    Two-stage retrieval:
    1. FAISS retrieves top_k candidates by embedding similarity.
    2. Cross-encoder reranker scores each (query, chunk) pair and
       returns the final_k most relevant chunks.

    This handles indirect / paraphrased / 'twisted' questions because
    the cross-encoder sees the full query+chunk text together, not just
    their independent embeddings.
    """
    query_emb = embedder.encode(
    [query], normalize_embeddings=True
    ).astype(np.float32)

   
    k = min(top_k, index.ntotal)
    D, I = index.search(query_emb, k)
    candidates = [all_chunks[i] for i in I[0]]

    if len(candidates) <= 1:
        return candidates

  
    pairs = [(query, c["code"]) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)

    return [c for _, c in ranked[:final_k]]


print(" Retrieval pipeline ready!")

 Retrieval pipeline ready!


In [14]:
def ask_question(query, show_context=False):
    top_chunks = retrieve_and_rerank(query, top_k=10, final_k=5)
    context = "\n\n".join([
        f"[{c['type'].upper()}] {c['name']}:\n{c['code']}"
        for c in top_chunks
    ])
    if show_context:
        print(" Retrieved Context:\n")
        print(context)
        print("\n" + "="*50 + "\n")

    system = """You are an expert code analyst.
Answer questions ONLY based on the provided code.
Be specific — reference exact function names, class names, and variable names.
If the user asks for the code of a function or class, reproduce it COMPLETELY and EXACTLY as given — do NOT omit, summarize, or say 'code omitted for brevity'.
If the answer is not in the code, say 'This is not covered in the provided code.'
Never hallucinate or invent information."""

    user = f"""Here is the relevant code from the project:

{context}

Question: {query}

Important: If the question asks for code of a function, copy the FULL function code from above — word for word, do not skip any lines, do not write '...' or 'omitted'.

Answer clearly and specifically, referencing the actual code above."""

    answer = call_mistral(system, user, max_tokens=800)
    return answer, top_chunks

In [15]:

def generate_code(description):
    """Generate new code matching the existing codebase style."""
    top_chunks = retrieve_and_rerank(description, top_k=8, final_k=3)

    context = "\n\n".join([
        f"[{c['type'].upper()}] {c['name']}:\n{c['code']}"
        for c in top_chunks
    ])

    system = f"""You are an expert {detected_lang} developer.
Generate clean, working code that matches the style and patterns of the existing codebase.
Always include docstrings/comments and type hints where appropriate.
Only generate what is asked — no extra explanation outside code comments."""

    user = f"""Here is the existing codebase for context and style reference:

{context}

Generate code for: {description}

Match the coding style, patterns, and conventions of the existing code above."""

    return call_mistral(system, user, max_tokens=1000)


In [16]:
def debug_function(function_name):
    """Deep debug a specific function by name."""
    chunk = next((c for c in all_chunks if c["name"] == function_name), None)

    if not chunk:
        print(f"'{function_name}' not found.")
        print("\n Available functions/classes:")
        for c in all_chunks:
            if c["type"] not in ("import", "text_chunk"):
                print(f"  [{c['type']}] {c['name']}")
        return

    system = """You are a senior code debugger and reviewer.
Analyze code deeply and provide specific, actionable fixes.
Always provide the corrected version of the code."""

    user = f"""Debug and review this {chunk['type']}:

{chunk['code']}

Provide:
1. Bugs & Logic Errors
2. Edge Cases & Missing Checks
3. Missing Error Handling
4. Performance Issues
5. Fixed & Improved Version of the code"""

    print(f"\n{'='*60}")
    print(f"Debug Report: `{function_name}`")
    print(f"{'='*60}\n")
    print(call_mistral(system, user, max_tokens=1200))
    print(f"\n{'='*60}\n")

In [17]:

def explain_line(function_name, line_number):
    """Explain a specific line within a named function."""
    chunk = next((c for c in all_chunks if c["name"] == function_name), None)

    if not chunk:
        print(f"'{function_name}' not found.")
        return

    lines = chunk['code'].split('\n')
    if line_number < 1 or line_number > len(lines):
        print(f" Line {line_number} out of range. Function has {len(lines)} lines.")
        return

    target_line = lines[line_number - 1]

    system = "You are an expert code teacher. Explain code clearly and simply."

    user = f"""In the function/class `{function_name}`:

Full code:
{chunk['code']}

Explain line {line_number} specifically:
{target_line}

Explain what this line does, why it's there, and how it fits in the overall context."""

    print(f"\n{'='*60}")
    print(f" Line {line_number} in `{function_name}`:")
    print(f"   {target_line.strip()}")
    print(f"{'='*60}\n")
    print(call_mistral(system, user, max_tokens=500))
    print(f"\n{'='*60}\n")


In [18]:
def summarize_file():
    named = sorted(
        [c for c in all_chunks if c["type"] not in ("import", "text_chunk")],
        key=lambda x: x["start"][0]
    )[:20]
    if not named:
        named = all_chunks[:20]

    snippets = "\n\n".join([
        f"[{c['type'].upper()}] {c['name']}:\n{c['code'][:300]}"
        for c in named
    ])

    system = "You are an expert code engineer. Summarize code accurately. Do not guess."

    user = f"""Here are the functions and classes from a {detected_lang} file:

{snippets}

Write a clear, accurate one-paragraph summary of what this file does.
Only describe what you actually see in the code."""

    return call_mistral(system, user, max_tokens=250)


print(" All functions ready!")

 All functions ready!


In [20]:

print(" RAG Code Assistant — Powered by Mistral-7B + Jina Embeddings")
print("="*65)
print("Commands:")
print("  'exit'                     → Quit")
print("  'list'                     → Show all functions/classes")
print("  'summary'                  → Summarize the file")
print("  'debug:<function_name>'    → Deep debug a function")
print("  'explain:<name>:<line>'    → Explain a specific line")
print("  'generate:<description>'   → Generate new code")
print("  or just ask any question!")
print("="*65 + "\n")

while True:
    user_query = input(" You: ").strip()

    if not user_query:
        continue

    if user_query.lower() == "exit":
        print(" Goodbye!")
        break

    elif user_query.lower() == "list":
        print("\n Available functions/classes:")
        for c in all_chunks:
            if c["type"] not in ("import", "text_chunk"):
                print(f"  [{c['type']}] {c['name']}")

    elif user_query.lower() == "summary":
        print("\n File Summary:\n")
        print(summarize_file())

    elif user_query.lower().startswith("debug:"):
        func_name = user_query.split("debug:", 1)[1].strip()
        debug_function(func_name)
    elif user_query.lower().startswith("explain:"):
        parts = user_query.split(":")
        if len(parts) == 3:
            func_name = parts[1].strip()
            try:
                line_num = int(parts[2].strip())
                explain_line(func_name, line_num)
            except ValueError:
                print(" Format: explain:function_name:line_number")
        else:
            print(" Format: explain:function_name:line_number")

    elif user_query.lower().startswith("generate:"):
        description = user_query.split("generate:", 1)[1].strip()
        print("\n Generating code...\n")
        code = generate_code(description)
        print(f"\n{'='*60}")
        print(" Generated Code:")
        print(f"{'='*60}\n")
        print(code)
        print(f"\n{'='*60}\n")

    else:
        answer, retrieved = ask_question(user_query)
        print("\n Retrieved Chunks (after reranking):")
        for r in retrieved:
            print(f"  [{r['type']}] {r['name']}")
        print(f"\n Answer:\n{answer}")

    print("\n" + "-"*65 + "\n")

 RAG Code Assistant — Powered by Mistral-7B + Jina Embeddings
Commands:
  'exit'                     → Quit
  'list'                     → Show all functions/classes
  'summary'                  → Summarize the file
  'debug:<function_name>'    → Deep debug a function
  'explain:<name>:<line>'    → Explain a specific line
  'generate:<description>'   → Generate new code
  or just ask any question!



 You:  summary


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



 File Summary:

This Python file contains various functions and classes used for parsing and analyzing source code. The main goal is to construct a directed graph representing the relationships between different entities such as files, classes, functions, variables, and method definitions. The file begins by defining utility functions like `get_code_text`, which retrieves the textual representation of a given code segment, and `add_node` and `add_edge`, which manage adding new nodes and edges to the graph respectively.

The core functionality includes `extract_chunks_and_graph`, which initializes an empty Directed Graph (DAG) and populates it with essential nodes like the 'FILE\_ROOT'. Functions like `handle_variable_reference` and `handle_expression` recursively traverse through the Abstract Syntax Tree (AST) to identify and add variable nodes along with their respective scopes. Additionally, there are functions for handling class definitions, creating a visualization of the graph us

 You:  list


...

 You:  So, what should i concluded with this project ?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



 Retrieved Chunks (after reranking):
  [import] import leidenalg
  [import] import heapq
  [import] from tree_sitter import Parser
  [import] from unixcoder import UniXcoder
  [function_definition] visualize_with_graphviz

 Answer:
This project appears to be about creating a graph representation of a codebase using various libraries such as Leidenalg, Tree Sitter, and PyDOT for graph visualization with Graphviz. The `visualize_with_graphviz` function takes a networkx graph object and saves its visualization as a PNG image named "code_graph.png". It also defines custom node and edge styles based on different types like files, classes, functions, imports, calls, variables, etc. Additionally, there are distinct styles for structural/definition edges, call edges, and data flow edges. The function attempts to save the generated graph as a PNG file upon successful execution.

-----------------------------------------------------------------



 You:  exit


 Goodbye!
